# Lekce 03 - Agentní návrhové vzory

V této lekci prozkoumáme tři základní návrhové vzory pro vytváření efektivních AI agentů:

1. **Jasné instrukce pro agenta** — Vytváření přesných, rolí definujících dotazů, které řídí chování agenta
2. **Strukturovaný výstup s modely Pydantic** — Zajištění, že agenti vrací předvídatelná, validovaná data
3. **Agent s jednou odpovědností** — Navrhování zaměřených agentů, kteří každý zvládnou jednu věc dobře

Každý vzor aplikujeme na scénář **doporučovače cestovních destinací**, postupně budujeme systém, který může navrhovat destinace, kontrolovat dostupnost a řešit logistiku.


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Vzor 1: Jasné pokyny pro agenta

Nejefektivnějším vzorem je také ten nejjednodušší: psaní jasných, podrobných pokynů pro vašeho agenta.

Dobré pokyny definují:
- **Kdo** agent je (persona a tón)
- **Co** by měl dělat (krok po kroku odpovědnosti)
- **Jak** by se měl chovat (omezení a styl)

Níže vytváříme cestovního concierge agenta s explicitními pokyny, které formují každou odpověď, kterou vytvoří.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzor 2: Strukturovaný výstup s Pydantic modely

Volný text je užitečný pro konverzaci, ale další systémy potřebují strukturovaná data.
Spárováním **Pydantic modelů** s **nástrojovou funkcí** můžeme:

- Definovat přesné schéma pro výstup agenta
- Automaticky validovat odpovědi
- Spolehlivě integrovat výsledky agenta do logiky aplikace

Klíčem k vynucení je předání `response_format` při spuštění agenta. To nutí
model vrátit validovaný objekt `TravelRecommendations` (dostupný na `response.value`)
místo volného textu. Nástroj `get_destination_details` také vrací typovaný
`DestinationRecommendation`, takže data zůstávají strukturovaná od začátku do konce.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Vzor 3: Agent s jednou odpovědností

Složité úkoly těží ze rozdělení práce mezi více zaměřených agentů, z nichž každý má jedinou odpovědnost:

- **Odborník na destinace**, který zná místa a dostupnost
- **Plánovač logistiky**, který řeší lety, hotely a itineráře

To odpovídá zásadě softwarového inženýrství *oddělení zájmů* — každý agent je snazší testovat, udržovat a nezávisle zlepšovat.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Shrnutí

V této lekci jsme aplikovali tři agentní návrhové vzory na scénář doporučování cestování:

| Vzor | Klíčová myšlenka | Přínos |
|---|---|---|
| **Jasné instrukce** | Definujte personu, odpovědnosti a omezení předem | Konzistentní, na značku laděné chování agenta |
| **Strukturovaný výstup** | Použijte Pydantic modely jako formát odpovědi | Validované, strojově čitelné výsledky |
| **Jedna odpovědnost** | Každému agentovi přiřaďte jedno zaměřené úkol | Snazší testování, údržba a skládání |

Tyto vzory se přirozeně kombinují — můžete spojit jasné instrukce se strukturovaným výstupem uvnitř agenta s jednou odpovědností a vytvořit tak robustní, produkčně připravené systémy.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
